# 09 — Similar-projects recommender

Ports and **finishes** `notebooks_original/recommender-system.ipynb`. That
notebook built three cosine-similarity matrices over the apartment *projects* in
`appartments.csv` and then left several contradictory weighted sums in place
(`30*sim1 + 20*sim2 + 8*sim3` in one cell, `6*sim1 + 5*sim2 + 3*sim3` in
another) with no conclusion.

**Scope.** This is a *"which developments are like this one"* feature
(item-to-item over 246 projects). The preference / budget / bedroom recommender
over individual listings is a separate, unbuilt component — see
`PROJECT_PLAN.md` §10.

**What this notebook does.** All logic lives in `src/recommender/`; this is the
thin runner. It shows (1) that the three matrices are on different scales,
(2) the chosen blend `structural 0.5 / location 0.3 / facilities 0.2` on
min-max-normalised matrices, (3) a sanity check on three known projects, and
(4) weight sensitivity. Full derivation: `reports/recommender/blend_weights.md`.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

from src.recommender.similarity import DEFAULT_WEIGHTS, build_components, load_appartments
from src.recommender.recommender import SimilarProjectsRecommender

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

raw = load_appartments()
raw_by_name = raw.set_index("PropertyName")
print(f"{len(raw)} projects   default weights: {DEFAULT_WEIGHTS}")

246 projects   default weights: {'structural': 0.5, 'location': 0.3, 'facilities': 0.2}


## 1. The three matrices are not on comparable scales

Cosine similarity is bounded, but the axes differ enough in spread and sign that
a naive weighted sum is dominated by `structural` on variance alone. This is why
each matrix is min-max normalised to [0, 1] before blending.

In [2]:
components = build_components(raw)

def offdiag_stats(m):
    n = m.shape[0]
    o = m[~np.eye(n, dtype=bool)]
    return {
        "min": o.min(), "max": o.max(), "mean": o.mean(), "std": o.std(),
        "pct_negative": (o < 0).mean(),
        "mean_top5_per_row": np.sort(m - np.eye(n) * 2, axis=1)[:, -5:].mean(),
    }

pd.DataFrame({k: offdiag_stats(v) for k, v in components.items()}).T.round(3)

,min,max,mean,std,pct_negative,mean_top5_per_row
facilities,0.000,0.683,0.072,0.082,0.000,0.365
structural,-0.766,1.000,0.033,0.341,0.546,0.855
location,-0.074,1.000,0.001,0.073,0.808,0.185


## 2. Sanity check — three known projects

`recommend()` returns the blended score plus the per-axis normalised similarity
for each pair, so the ranking is explainable.

In [3]:
rec = SimilarProjectsRecommender.from_csv()  # default 0.5 / 0.3 / 0.2

TARGETS = ["DLF The Arbour", "M3M Golf Hills", "Ireo Victory Valley"]
for t in TARGETS:
    print("=" * 90)
    print(f"QUERY: {t}  |  {raw_by_name.loc[t, 'PropertySubName']}")
    out = rec.recommend(t, k=5)
    for _, r in out.iterrows():
        sub = raw_by_name.loc[r["PropertyName"], "PropertySubName"]
        print(f"  {r['rank']}. {r['PropertyName']:<32} score={r['score']:.3f}  "
              f"[struct={r['structural']:.2f} loc={r['location']:.2f} fac={r['facilities']:.2f}]  {sub}")
    print()

QUERY: DLF The Arbour  |  4 BHK Apartment in Sector 63 Gurgaon
  1. DLF The Summit                   score=0.546  [struct=0.99 loc=0.06 fac=0.16]  4 BHK Apartment in Sector 54, Gurgaon
  2. DLF The Pinnacle                 score=0.520  [struct=1.00 loc=0.06 fac=0.01]  4 BHK Apartment in DLF Phase 5, Gurgaon
  3. Paras Quartier                   score=0.519  [struct=0.98 loc=0.08 fac=0.02]  4 BHK Apartment in Gwal Pahari, Gurgaon
  4. Tulip Purple                     score=0.514  [struct=0.85 loc=0.06 fac=0.35]  4 BHK Apartment in Sector 69, Gurgaon
  5. BPTP Mansions Park Prime         score=0.511  [struct=0.87 loc=0.06 fac=0.29]  4 BHK Apartment in Sector 66, Gurgaon

QUERY: M3M Golf Hills  |  2, 3, 4 BHK Apartment in Sector 79, Gurgaon
  1. BPTP Terra                       score=0.551  [struct=0.97 loc=0.12 fac=0.15]  2, 3, 4 BHK Apartment in Sector 37D, Gurgaon
  2. Corona Optus                     score=0.547  [struct=0.95 loc=0.06 fac=0.25]  2, 3, 4 BHK Apartment in Sector 37C, Gu

## 3. Weight sensitivity

The chosen weights should not be knife-edge, and should beat both equal-weighting
and the original notebook's ratio.

In [4]:
ALTS = {
    "0.5/0.3/0.2 (chosen)":   {"structural": .5,   "location": .3,   "facilities": .2},
    "0.4/0.4/0.2":            {"structural": .4,   "location": .4,   "facilities": .2},
    "0.6/0.2/0.2":            {"structural": .6,   "location": .2,   "facilities": .2},
    "equal (1/3 each)":       {"structural": 1/3,  "location": 1/3,  "facilities": 1/3},
    "notebook 6/5/3 -> norm": {"structural": 5/14, "location": 3/14, "facilities": 6/14},
}
for label, w in ALTS.items():
    r = SimilarProjectsRecommender(raw, weights=w)
    print(label)
    for t in TARGETS:
        print(f"  {t:<21} -> {r.recommend(t, k=5)['PropertyName'].tolist()}")
    print()

0.5/0.3/0.2 (chosen)
  DLF The Arbour        -> ['DLF The Summit', 'DLF The Pinnacle', 'Paras Quartier', 'Tulip Purple', 'BPTP Mansions Park Prime']
  M3M Golf Hills        -> ['BPTP Terra', 'Corona Optus', 'Unitech Escape', 'Puri Emerald Bay', 'Mahindra Aura']
  Ireo Victory Valley   -> ['Ambience Creacions', 'Pioneer Urban Presidia', 'Pioneer Araya', 'Bestech Park View Grand Spa', 'DLF The Crest']



0.4/0.4/0.2
  DLF The Arbour        -> ['DLF The Summit', 'Tulip Purple', 'BPTP Mansions Park Prime', 'Paras Quartier', 'DLF The Pinnacle']
  M3M Golf Hills        -> ['BPTP Terra', 'Unitech Escape', 'Corona Optus', 'Puri Emerald Bay', 'Mahindra Aura']
  Ireo Victory Valley   -> ['Ambience Creacions', 'Pioneer Urban Presidia', 'Bestech Park View Grand Spa', 'Pioneer Araya', 'DLF The Crest']



0.6/0.2/0.2
  DLF The Arbour        -> ['DLF The Summit', 'DLF The Pinnacle', 'Paras Quartier', 'Tulip Purple', 'BPTP Mansions Park Prime']
  M3M Golf Hills        -> ['Corona Optus', 'BPTP Terra', 'Unitech Escape', 'Puri Emerald Bay', 'Unitech Harmony']
  Ireo Victory Valley   -> ['Ambience Creacions', 'Pioneer Urban Presidia', 'Pioneer Araya', 'Bestech Park View Grand Spa', 'DLF The Crest']



equal (1/3 each)
  DLF The Arbour        -> ['JMS The Nation', 'Oxirich Chintamanis', 'Tulip Purple', 'Vatika Aspiration', 'BPTP Mansions Park Prime']
  M3M Golf Hills        -> ['Ashiana Amarah', 'Corona Optus', 'Puri Emerald Bay', 'BPTP Terra', 'Mahindra Aura']
  Ireo Victory Valley   -> ['Pioneer Urban Presidia', 'Ambience Creacions', 'Silverglades The Melia', 'DLF The Crest', 'AIPL The Peaceful Homes']



notebook 6/5/3 -> norm
  DLF The Arbour        -> ['JMS The Nation', 'Oxirich Chintamanis', 'Vatika Aspiration', 'Tulip Purple', 'SS Linden Floors']
  M3M Golf Hills        -> ['Ashiana Amarah', 'Corona Optus', 'Puri Emerald Bay', 'Godrej Nature Plus Serenity', 'Mahindra Aura']
  Ireo Victory Valley   -> ['Pioneer Urban Presidia', 'Silverglades The Melia', 'DLF The Crest', 'Ambience Creacions', 'AIPL The Peaceful Homes']



## Takeaway

- Blend = `structural 0.5 / location 0.3 / facilities 0.2` on min-max-normalised
  matrices. Reasoning, scale table, and known limitations:
  `reports/recommender/blend_weights.md`.
- `structural` (BHK config / area / price band) carries the ranking; `facilities`
  is a tie-breaker; `location` scored above 0.2 in only 1 of the 15
  sanity-check recommendations — at effective weight it is closer to a rare
  tiebreak than a 30 % input, kept for when it does fire and for when a denser
  location signal (`sector`) replaces the sparse landmark distances.
- Listing-level preference matching is out of scope here — `PROJECT_PLAN.md` §10.


---

# Landmark-proximity search

A second, independent feature for the app's recommender page: pick a named
landmark, get every project within X km, nearest first. Stateless lookup over a
precomputed **projects × landmarks distance matrix** (metres) built from the
same `LocationAdvantages` data that feeds `cosine_sim3` above — reshaped into raw
distances instead of a similarity score. Code: `src/recommender/landmark_search.py`.

**Limitation (must surface in the app UI — `PROJECT_PLAN.md` §12):** results are
*projects whose listing mentions this landmark within the radius*, not a true
geospatial nearest. `appartments.csv` has no project coordinates, so a project
that simply didn't list a landmark is absent even if it is physically close.

In [5]:
from src.recommender.landmark_search import (
    build_distance_matrix, search_by_landmark, list_landmarks,
)

dist = build_distance_matrix()   # also writes data/processed/landmark_distance_matrix.csv
no_data = int((~dist.notna().any(axis=1)).sum())
print(f"{dist.shape[0]} projects x {dist.shape[1]} landmarks | "
      f"{int(dist.notna().sum().sum())} known distances | "
      f"{no_data} projects with no parseable landmark distance (all-NaN rows)")
print("landmarks listed by >= 10 projects:", len(list_landmarks(dist, min_projects=10)))

246 projects x 965 landmarks | 2370 known distances | 17 projects with no parseable landmark distance (all-NaN rows)
landmarks listed by >= 10 projects: 30


In [6]:
for lm, radius in [("Dwarka Expressway", 2), ("Sector 55-56 Metro Station", 5),
                   ("Indira Gandhi International Airport", 18)]:
    hits = search_by_landmark(lm, radius, matrix=dist)
    print(f"\n{lm} — within {radius} km: {len(hits)} projects")
    print(hits.head(6).to_string(index=False))


Dwarka Expressway — within 2 km: 21 projects
               project  distance_m  distance_km
           M3M Capital          10         0.01
   Satya Merano Greens          75         0.08
     Indiabulls Enigma         100         0.10
Puri Diplomatic Greens         300         0.30
      Puri Emerald Bay         350         0.35
             M3M Crown         450         0.45

Sector 55-56 Metro Station — within 5 km: 10 projects
               project  distance_m  distance_km
    Puri The Aravallis        2700          2.7
     Mahindra Luminare        2700          2.7
     Anant Raj Estates        3600          3.6
Anant Raj Ashok Estate        3700          3.7
   Adani Samsara Avasa        3800          3.8
       Emaar Digihomes        3900          3.9

Indira Gandhi International Airport — within 18 km: 26 projects
                 project  distance_m  distance_km
      Ambience Creacions       11000         11.0
               M3M Crown       14100         14.1
Emaar MGF Em

In [7]:
# explicit edge-case handling the reference page lacked
empty = search_by_landmark("Sector 55-56 Metro Station", 2, matrix=dist)  # nearest is 2.7 km
print("no match within radius -> empty frame, normal columns:",
      empty.empty, list(empty.columns))

for bad in (0, -3):
    try:
        search_by_landmark("Dwarka Expressway", bad, matrix=dist)
    except ValueError as e:
        print(f"radius_km={bad} -> ValueError: {e}")

try:
    search_by_landmark("Buckingham Palace", 5, matrix=dist)
except KeyError as e:
    print("unknown landmark ->", str(e)[:90], "...")

no match within radius -> empty frame, normal columns: True ['project', 'distance_m', 'distance_km']
radius_km=0 -> ValueError: radius_km must be > 0, got 0
radius_km=-3 -> ValueError: radius_km must be > 0, got -3
unknown landmark -> "landmark not found: 'Buckingham Palace'. Did you mean: ['Neemrana Palace', 'Hyatt Place', ...


**Reading it.** `Dwarka Expressway` @ 2 km returns 21 projects, several
essentially on the road (M3M Capital 10 m, Satya Merano Greens 75 m). The metro
station's nearest project is 2.7 km, so a 2 km search correctly returns nothing —
an **empty DataFrame with the normal columns**, not an error and not a silent
`None`. `radius_km <= 0` raises `ValueError` (km in, never metres — no
off-by-1000). `IGI Airport` folds to `Indira Gandhi International Airport` via a
small alias map; the long tail of name variants is not resolved.

---

# Preference-based listing recommender

The third app feature and the one with no reference to port. Takes a **preference
vector** over individual listings (`gurgaon_properties_missing_value_imputation.csv`),
applies **hard filters** (budget ceiling, min bedrooms, sector, property type),
then ranks the survivors by **weighted closeness** on the soft axes the user
specified: `built_up_area` 0.35 / `luxury_score` 0.25 / `bathroom` 0.15 /
`agePossession` 0.15 / `furnishing_type` 0.10, each scaled by its own
IQR-or-std over the full table. The price model and project-similarity are
**display columns only**, never in the ranking score. Design + the two deliberate
scope choices (luxury weight vs price-importance; sector as a hard filter only):
`reports/recommender/listing_recommender_design.md`.

In [8]:
from src.recommender.listing_recommender import ListingRecommender, Preferences

listings = ListingRecommender.from_csv()
print("axis scales:", {k: round(v, 1) for k, v in listings._scale.items()})
print("society -> project bridge:", len(listings._society_to_project), "societies")

VIEW = ["society", "sector", "property_type", "price", "bedRoom", "bathroom",
        "built_up_area", "agePossession", "furnishing_type", "match_score",
        "predicted_price_cr", "price_vs_model_pct", "similar_projects"]

axis scales: {'area_sqft': 997.5, 'luxury_score': 77.0, 'bathrooms': 1.5, 'age_possession': 1.0, 'furnishing': 0.6}
society -> project bridge: 172 societies


### A — 3+ BHK flat, ≤ ₹2.5 Cr, ~1600 sqft, semi-furnished, Relatively New

In [9]:
a = listings.recommend(Preferences(
    budget_max_cr=2.5, min_bedrooms=3, property_type="flat",
    area_sqft=1600, furnishing="semifurnished", age_possession="Relatively New",
), k=6)
a[VIEW]

,society,sector,property_type,price,bedRoom,bathroom,built_up_area,agePossession,furnishing_type,match_score,predicted_price_cr,price_vs_model_pct,similar_projects
0,emaar mgf emerald floors premier,sector 65,flat,2.25,3.0,3.0,1600.0,Relatively New,1.0,0.0,2.32,-3.0,"Adani Brahma Samsara, Optimal ultra luxury bui..."
1,vatika gurgaon,sector 83,flat,1.10,3.0,3.0,1600.0,Relatively New,1.0,0.0,1.21,-9.1,None
2,emaar palm gardens,sector 83,flat,1.75,3.0,3.0,1600.0,Relatively New,1.0,0.0,1.73,1.2,"Tulip Yellow, Orchid Petals, Adani M2K Oyster ..."
3,emaar palm gardens,sector 83,flat,1.72,3.0,3.0,1600.0,Relatively New,1.0,0.0,1.73,-0.6,"Tulip Yellow, Orchid Petals, Adani M2K Oyster ..."
4,emaar mgf emerald floors premier,sector 65,flat,2.40,3.0,3.0,1600.0,Relatively New,1.0,0.0,2.32,3.4,"Adani Brahma Samsara, Optimal ultra luxury bui..."
5,emaar palm gardens,sector 83,flat,1.75,3.0,3.0,1600.0,Relatively New,1.0,0.0,1.73,1.2,"Tulip Yellow, Orchid Petals, Adani M2K Oyster ..."


### B — 2+ BHK, ≤ ₹1 Cr, in sector 92, ~1200 sqft (hard-filter heavy)

In [10]:
b = listings.recommend(Preferences(
    budget_max_cr=1.0, min_bedrooms=2, sector="sector 92", area_sqft=1200,
), k=6)
b[VIEW]

,society,sector,property_type,price,bedRoom,bathroom,built_up_area,agePossession,furnishing_type,match_score,predicted_price_cr,price_vs_model_pct,similar_projects
0,ansal heights,sector 92,flat,0.60,2.0,2.0,1195.00,Under Construction,0.0,0.005013,0.75,-20.0,None
1,parkwood westend,sector 92,flat,0.70,2.0,2.0,1217.00,Under Construction,0.0,0.017043,0.78,-10.3,None
2,sare homes,sector 92,flat,0.82,3.0,3.0,1221.18,Relatively New,0.0,0.021233,0.82,0.0,None
3,sare crescent parc royal greens phase 1,sector 92,flat,0.79,3.0,2.0,1175.00,New Property,0.0,0.025063,0.80,-1.2,None
4,sare green parc phase 3,sector 92,flat,0.79,3.0,2.0,1175.00,New Property,0.0,0.025063,0.80,-1.2,None
5,raheja sampada,sector 92,flat,0.65,3.0,2.0,1240.00,Relatively New,0.0,0.040100,0.86,-24.4,None


### C — no match: 4+ BHK flat under ₹0.5 Cr

In [11]:
c_res = listings.recommend(Preferences(
    budget_max_cr=0.5, min_bedrooms=4, property_type="flat",
), k=6)
print("empty:", c_res.empty, "| rows:", len(c_res), "| columns:", list(c_res.columns))

empty: True | rows: 0 | columns: ['society', 'sector', 'property_type', 'price', 'price_per_sqft', 'bedRoom', 'bathroom', 'built_up_area', 'agePossession', 'furnishing_type', 'floorNum', 'luxury_score', 'match_score', 'predicted_price_cr', 'price_vs_model_pct', 'similar_projects']


### D — hard filters only, no soft axis: `match_score` is NaN, sorted by price

In [12]:
d = listings.recommend(Preferences(budget_max_cr=1.2, min_bedrooms=2), k=5)
print("match_score all NaN:", d["match_score"].isna().all())
d[VIEW]

match_score all NaN: True


,society,sector,property_type,price,bedRoom,bathroom,built_up_area,agePossession,furnishing_type,match_score,predicted_price_cr,price_vs_model_pct,similar_projects
0,ashiana apartment,sector 23,flat,0.16,2.0,2.0,706.0,Moderately Old,0.0,NaN,0.35,-54.3,None
1,hcbs sports ville,sohna road,flat,0.20,2.0,2.0,743.0,New Property,0.0,NaN,0.28,-28.6,None
2,vidya apartment,sector 5,flat,0.22,2.0,2.0,570.0,Relatively New,2.0,NaN,0.30,-26.7,None
3,independent,sector 9,house,0.22,2.0,2.0,37.0,Old Property,0.0,NaN,0.46,-52.2,None
4,city shri ram apartments 1,sector 110,flat,0.22,2.0,1.0,543.0,Relatively New,0.0,NaN,0.29,-24.1,None


**Reading it.** (A) every result is a 3 BHK / 1600 sqft / Relatively New /
semi-furnished flat — `match_score` 0 means the specified axes matched exactly;
`price_vs_model_pct` shows each one priced within a few percent of the model.
(B) the `sector 92` hard filter binds; `area_sqft` is the only soft axis, so
ranking is by area closeness. (C) an impossible combination returns an **empty
DataFrame with the full column set**, not an error. (D) with no soft preference,
`match_score` is `NaN` and results come back cheapest-first. Note the source
table has near-duplicate listings (same unit type, slightly different price) —
not de-duplicated in v1 (`listing_recommender_design.md`).

## Recommender features — status

| Feature | Module | State |
|---|---|---|
| Similar developments (item-to-item) | `SimilarProjectsRecommender` | done |
| Landmark-proximity search | `landmark_search` | done |
| Preference-based listing recommender | `ListingRecommender` | done |

All three feed the app's recommender page (`PROJECT_PLAN.md` §12).